# Distillation sérieuse — teacher Evo2 out-of-fold vers un petit MLP

## Question expérimentale

Peut-on dépasser **90 % d'Accuracy** avec seulement :

- les **4 000** fenêtres train associées aux embeddings Evo2 ;
- les **1 000** fenêtres de validation Evo2 pour l'évaluation finale ;
- un student de **48 961 paramètres**, ne dépendant pas d'Evo2 à l'inférence ?

Le point méthodologique central est que chaque cible teacher utilisée par le student doit être produite par un teacher qui **n'a jamais vu cet exemple ni son organisme** pendant son entraînement.

## Protocole anti-fuite

```text
4 000 exemples train, 20 organismes
            │
            ├── 3 246 exemples de développement, 16 organismes
            │       └── teachers croisés → logits OOF
            │
            └── 754 exemples de réglage, 4 organismes
                    └── choix de alpha, T, époque et seuil

Choix figés
            ↓
5 teachers croisés sur les 4 000 exemples → 4 000 logits OOF
            ↓
student final entraîné sur 4 000 exemples
            ↓
évaluation unique sur les 1 000 validations officielles
```

Les splits sont groupés par `organism`. Le test n'est jamais chargé. La validation officielle ne choisit aucun hyperparamètre.

In [1]:
import copy
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

SEED = 42
TEACHER_EPOCHS = 40
STUDENT_MAX_EPOCHS = 80
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

CWD = Path.cwd().resolve()
DAY3_DIR = CWD if (CWD / "src").exists() else CWD / "day3"
if not (DAY3_DIR / "src" / "advanced_features.py").exists():
    raise FileNotFoundError("Lancez ce notebook depuis la racine du projet ou day3/.")
ROOT = DAY3_DIR.parent
sys.path.insert(0, str(DAY3_DIR / "src"))

from advanced_features import advanced_feature_matrix
from embeddings import load_supervised_embeddings
from models.classifier_heads import MLPHead

print("Device :", DEVICE)
print("Teacher epochs :", TEACHER_EPOCHS)

Device : mps
Teacher epochs : 40


## 1. Charger uniquement les IDs possédant un embedding Evo2

Nous ne chargeons pas les 58 552 lignes du train complet. Les IDs contenus dans les fichiers `.npz` définissent strictement notre expérience : 4 000 train et 1 000 validation, équilibrés entre les deux classes.

In [2]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "mps":
        torch.mps.manual_seed(seed)

def load_aligned(split):
    embeddings, labels, ids = load_supervised_embeddings(ROOT / "2-data/embeddings", split)
    frame = (
        pd.read_csv(ROOT / f"2-data/processed/{split}.csv", dtype={"sequence": str})
        .set_index("id").loc[ids].reset_index()
    )
    assert np.array_equal(frame["label"].to_numpy(), labels)
    assert frame["sequence"].str.len().eq(200).all()
    return frame, embeddings.astype(np.float32), labels.astype(np.int64)

train_df, train_embeddings, y_train = load_aligned("train")
val_df, _, y_val = load_aligned("val")
groups = train_df["organism"].to_numpy()

display(pd.DataFrame([
    {"split": "train", "N": len(train_df), "classe 0": int((y_train == 0).sum()), "classe 1": int((y_train == 1).sum()), "organismes": train_df.organism.nunique()},
    {"split": "validation finale", "N": len(val_df), "classe 0": int((y_val == 0).sum()), "classe 1": int((y_val == 1).sum()), "organismes": val_df.organism.nunique()},
]))

,split,N,classe 0,classe 1,organismes
0,train,4000,2000,2000,20
1,validation finale,1000,500,500,4


## 2. Entrées du student

Le teacher reçoit les embeddings Evo2 de dimension 4 096. Le student ne les reçoit jamais : il utilise les 761 caractéristiques codon/phase calculées directement depuis l'ADN. À l'inférence, Evo2 peut donc être retiré.

In [3]:
feature_start = time.perf_counter()
X_train = advanced_feature_matrix(train_df["sequence"].tolist())
X_val = advanced_feature_matrix(val_df["sequence"].tolist())
print("Student train :", X_train.shape)
print("Student validation :", X_val.shape)
print(f"Extraction : {time.perf_counter() - feature_start:.2f} s")

Student train : (4000, 761)
Student validation : (1000, 761)
Extraction : 2.05 s


## 3. Produire des logits teacher out-of-fold

Pour chaque fold, une nouvelle tête teacher est entraînée sur 16 organismes et prédit les quatre organismes restants. Ainsi, chaque logit utilisé par le student provient d'un teacher qui n'a jamais vu cet exemple ni son organisme.

In [4]:
def train_teacher_predict(embeddings, labels, train_idx, predict_idx, seed):
    seed_everything(seed)
    teacher = MLPHead(d_in=embeddings.shape[1], d_hidden=128, dropout=0.1).to(DEVICE)
    optimizer = torch.optim.AdamW(teacher.parameters(), lr=1e-3, weight_decay=1e-4)
    x = torch.from_numpy(embeddings)
    y = torch.tensor(labels, dtype=torch.float32)
    generator = torch.Generator().manual_seed(seed)

    for _ in range(TEACHER_EPOCHS):
        teacher.train()
        permutation = train_idx[torch.randperm(len(train_idx), generator=generator).numpy()]
        for start in range(0, len(permutation), 256):
            idx = permutation[start:start + 256]
            optimizer.zero_grad(set_to_none=True)
            logits = teacher(x[idx].to(DEVICE))
            loss = F.binary_cross_entropy_with_logits(logits, y[idx].to(DEVICE))
            loss.backward()
            optimizer.step()

    teacher.eval()
    with torch.no_grad():
        return teacher(x[predict_idx].to(DEVICE)).cpu().numpy()

def cross_fitted_logits(embeddings, labels, organisms, n_splits, base_seed):
    result = np.full(len(labels), np.nan, dtype=np.float32)
    rows = []
    splitter = GroupKFold(n_splits=n_splits)
    for fold, (fit_idx, held_idx) in enumerate(splitter.split(embeddings, labels, organisms), 1):
        fold_logits = train_teacher_predict(embeddings, labels, fit_idx, held_idx, base_seed + fold)
        result[held_idx] = fold_logits
        prediction = (fold_logits >= 0).astype(int)
        rows.append({
            "fold": fold, "train N": len(fit_idx), "OOF N": len(held_idx),
            "organismes OOF": len(np.unique(organisms[held_idx])),
            "Accuracy": accuracy_score(labels[held_idx], prediction),
            "F1": f1_score(labels[held_idx], prediction),
        })
    assert np.isfinite(result).all()
    return result, pd.DataFrame(rows)

## 4. Réglage interne, groupé par organisme

Nous réservons quatre organismes parmi les 20 organismes train. Ce holdout interne choisit `alpha`, la température, le nombre d'époques et le seuil de décision. Les 1 000 validations officielles restent fermées.

In [5]:
internal_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
dev_idx, tune_idx = next(internal_split.split(X_train, y_train, groups))
assert not (set(groups[dev_idx]) & set(groups[tune_idx]))

dev_oof_logits, dev_teacher_df = cross_fitted_logits(
    train_embeddings[dev_idx], y_train[dev_idx], groups[dev_idx], n_splits=4, base_seed=100
)
print(f"Développement : {len(dev_idx)} exemples, {len(np.unique(groups[dev_idx]))} organismes")
print(f"Réglage interne : {len(tune_idx)} exemples, {len(np.unique(groups[tune_idx]))} organismes")
display(dev_teacher_df.style.format({"Accuracy": "{:.2%}", "F1": "{:.2%}"}))

Développement : 3246 exemples, 16 organismes
Réglage interne : 754 exemples, 4 organismes


,fold,train N,OOF N,organismes OOF,Accuracy,F1
0,1,2496,750,4,98.27%,98.26%
1,2,2386,860,4,97.91%,97.82%
2,3,2379,867,4,97.46%,97.47%
3,4,2477,769,4,98.83%,98.84%


## 5. Petit student et perte de distillation

Le student possède 761 entrées, une couche cachée de 64 neurones et une sortie binaire : 48 961 paramètres.

La perte est :

$$L = \alpha L_{hard} + (1-\alpha)T^2 L_{soft}$$

`alpha=1` est le témoin sans distillation. Les autres variantes imitent partiellement les logits teacher OOF.

In [6]:
class SmallStudent(nn.Module):
    def __init__(self, d_in=761):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def standardize_from_train(x_train, x_other):
    mean, std = x_train.mean(0, keepdims=True), x_train.std(0, keepdims=True)
    std[std < 1e-7] = 1.0
    return (x_train - mean) / std, (x_other - mean) / std

def best_threshold(labels, probabilities):
    candidates = np.linspace(0.25, 0.75, 201)
    scored = [(
        accuracy_score(labels, probabilities >= t),
        f1_score(labels, probabilities >= t), -abs(t - 0.5), t
    ) for t in candidates]
    return float(max(scored)[-1])

def train_student(x_train_np, labels, teacher_logits_np, x_eval_np, eval_labels,
                  alpha, temperature, seed, epochs, tune=False, threshold=0.5,
                  return_model=False):
    seed_everything(seed)
    model = SmallStudent(x_train_np.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-3)
    generator = torch.Generator().manual_seed(seed)
    x = torch.tensor(x_train_np, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.float32)
    teacher_logits = torch.tensor(teacher_logits_np, dtype=torch.float32)
    x_eval = torch.tensor(x_eval_np, dtype=torch.float32)
    best = None

    for epoch in range(1, epochs + 1):
        model.train()
        permutation = torch.randperm(len(x), generator=generator)
        for start in range(0, len(x), 256):
            idx = permutation[start:start + 256]
            student_logits = model(x[idx].to(DEVICE))
            hard = F.binary_cross_entropy_with_logits(student_logits, y[idx].to(DEVICE))
            soft = F.binary_cross_entropy_with_logits(
                student_logits / temperature,
                torch.sigmoid(teacher_logits[idx].to(DEVICE) / temperature),
            ) * temperature**2
            loss = alpha * hard + (1 - alpha) * soft
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        if tune or epoch == epochs:
            model.eval()
            with torch.no_grad():
                logits = model(x_eval.to(DEVICE)).cpu().numpy()
            probabilities = 1 / (1 + np.exp(-logits))
            current_threshold = best_threshold(eval_labels, probabilities) if tune else threshold
            prediction = probabilities >= current_threshold
            row = {
                "epoch": epoch, "threshold": current_threshold,
                "Accuracy": accuracy_score(eval_labels, prediction),
                "F1": f1_score(eval_labels, prediction),
                "ROC-AUC": roc_auc_score(eval_labels, probabilities),
            }
            if best is None or (row["Accuracy"], row["F1"]) > (best["Accuracy"], best["F1"]):
                best = row
    return (best, model) if return_model else best

print("Paramètres student :", sum(p.numel() for p in SmallStudent().parameters()))

Paramètres student : 48961


## 6. Choisir la configuration sans ouvrir la validation finale

In [7]:
X_dev, X_tune = standardize_from_train(X_train[dev_idx], X_train[tune_idx])
candidates = [
    ("hard", 1.0, 2.0),
    ("KD a=0.8 T=1", 0.8, 1.0),
    ("KD a=0.5 T=1", 0.5, 1.0),
    ("KD a=0.8 T=2", 0.8, 2.0),
    ("KD a=0.5 T=2", 0.5, 2.0),
    ("KD a=0.8 T=4", 0.8, 4.0),
]
tuning_rows = []
for name, alpha, temperature in candidates:
    result = train_student(
        X_dev, y_train[dev_idx], dev_oof_logits, X_tune, y_train[tune_idx],
        alpha, temperature, SEED, STUDENT_MAX_EPOCHS, tune=True,
    )
    tuning_rows.append({"configuration": name, "alpha": alpha, "T": temperature, **result})

tuning_df = pd.DataFrame(tuning_rows)
display(tuning_df.style.format({"Accuracy": "{:.2%}", "F1": "{:.2%}", "ROC-AUC": "{:.4f}"}))
selected_hard = tuning_df[tuning_df.configuration == "hard"].iloc[0]
selected_kd = tuning_df[tuning_df.configuration != "hard"].sort_values(["Accuracy", "F1"], ascending=False).iloc[0]
print("Témoin choisi :", selected_hard.to_dict())
print("KD choisie :", selected_kd.to_dict())

,configuration,alpha,T,epoch,threshold,Accuracy,F1,ROC-AUC
0,hard,1.000000,2.000000,55,0.255000,91.64%,92.12%,0.9638
1,KD a=0.8 T=1,0.800000,1.000000,71,0.270000,92.71%,93.22%,0.9666
2,KD a=0.5 T=1,0.500000,1.000000,57,0.265000,93.24%,93.70%,0.9687
3,KD a=0.8 T=2,0.800000,2.000000,71,0.287500,92.97%,93.45%,0.9667
4,KD a=0.5 T=2,0.500000,2.000000,47,0.282500,92.44%,92.99%,0.9651
5,KD a=0.8 T=4,0.800000,4.000000,47,0.257500,91.78%,92.33%,0.9633


Témoin choisi : {'configuration': 'hard', 'alpha': 1.0, 'T': 2.0, 'epoch': 55, 'threshold': 0.255, 'Accuracy': 0.916445623342175, 'F1': 0.9211514392991239, 'ROC-AUC': 0.9638306124607736}
KD choisie : {'configuration': 'KD a=0.5 T=1', 'alpha': 0.5, 'T': 1.0, 'epoch': 57, 'threshold': 0.265, 'Accuracy': 0.9323607427055703, 'F1': 0.9369592088998764, 'ROC-AUC': 0.9686964493494588}


## 7. Générer les 4 000 cibles OOF finales

Les choix étant maintenant figés, cinq teachers temporaires produisent un logit OOF pour chacun des 4 000 exemples.

In [8]:
all_oof_logits, teacher_oof_df = cross_fitted_logits(
    train_embeddings, y_train, groups, n_splits=5, base_seed=200
)
display(teacher_oof_df.style.format({"Accuracy": "{:.2%}", "F1": "{:.2%}"}))
print("Teacher OOF Accuracy globale :", f"{accuracy_score(y_train, all_oof_logits >= 0):.2%}")
print("Chaque exemple possède exactement un logit OOF :", np.isfinite(all_oof_logits).all())

,fold,train N,OOF N,organismes OOF,Accuracy,F1
0,1,3253,747,4,97.72%,97.74%
1,2,3149,851,4,98.82%,98.75%
2,3,3149,851,4,97.18%,97.22%
3,4,3237,763,4,98.30%,98.35%
4,5,3212,788,4,96.70%,96.81%


Teacher OOF Accuracy globale : 97.75%
Chaque exemple possède exactement un logit OOF : True


## 8. Entraînement final et ouverture de la validation officielle

Nous réentraînons le témoin et la KD sur les 4 000 exemples avec trois graines déclarées à l'avance. Les époques, seuils, `alpha` et `T` viennent uniquement du réglage interne.

In [9]:
X_all, X_final_val = standardize_from_train(X_train, X_val)
final_rows = []
for seed in (7, 42, 123):
    for kind, config in (("Sans distillation", selected_hard), ("Distillation OOF", selected_kd)):
        result = train_student(
            X_all, y_train, all_oof_logits, X_final_val, y_val,
            float(config["alpha"]), float(config["T"]), seed, int(config["epoch"]),
            tune=False, threshold=float(config["threshold"]),
        )
        final_rows.append({"student": kind, "seed": seed, **result})

final_df = pd.DataFrame(final_rows)
display(final_df.style.format({"Accuracy": "{:.2%}", "F1": "{:.2%}", "ROC-AUC": "{:.4f}"}))
summary_df = final_df.groupby("student").agg(
    accuracy_moyenne=("Accuracy", "mean"),
    accuracy_ecart_type=("Accuracy", lambda values: values.std(ddof=0)),
    f1_moyen=("F1", "mean"),
    roc_auc_moyenne=("ROC-AUC", "mean"),
).reset_index()
display(summary_df.style.format({
    "accuracy_moyenne": "{:.2%}", "accuracy_ecart_type": "{:.2%}",
    "f1_moyen": "{:.2%}", "roc_auc_moyenne": "{:.4f}",
}))

,student,seed,epoch,threshold,Accuracy,F1,ROC-AUC
0,Sans distillation,7,55,0.255000,89.20%,89.62%,0.9535
1,Distillation OOF,7,57,0.265000,90.20%,90.56%,0.9643
2,Sans distillation,42,55,0.255000,89.80%,90.19%,0.9530
3,Distillation OOF,42,57,0.265000,90.30%,90.68%,0.9621
4,Sans distillation,123,55,0.255000,90.40%,90.73%,0.9525
5,Distillation OOF,123,57,0.265000,89.90%,90.32%,0.9624


,student,accuracy_moyenne,accuracy_ecart_type,f1_moyen,roc_auc_moyenne
0,Distillation OOF,90.13%,0.17%,90.52%,0.9629
1,Sans distillation,89.80%,0.49%,90.18%,0.9530


## 9. Taille et latence complète du student OOF

La latence de déploiement comprend le calcul des 761 caractéristiques puis le `forward` du petit MLP. Pour rester comparable au student original, les deux modèles sont mesurés sur **CPU**, avec un batch de taille 1 pour le réseau. Evo2 et les teachers temporaires ne sont pas inclus : ils n'existent plus dans le chemin d'inférence.

Le modèle de la graine 42 est reconstruit avec les hyperparamètres déjà figés. L'Accuracy et le F1 affichés dans le benchmark restent les moyennes sur les trois graines.

In [10]:
from benchmark import (
    benchmark_feature_extraction, benchmark_torch_forward,
    count_trainable_parameters, model_storage_mb,
)

benchmark_result, benchmark_student = train_student(
    X_all, y_train, all_oof_logits, X_final_val, y_val,
    float(selected_kd["alpha"]), float(selected_kd["T"]),
    seed=42, epochs=int(selected_kd["epoch"]),
    tune=False, threshold=float(selected_kd["threshold"]),
    return_model=True,
)

oof_feature_ms = benchmark_feature_extraction(
    advanced_feature_matrix, val_df["sequence"].tolist(), repeats=5,
)
oof_forward_ms = benchmark_torch_forward(
    benchmark_student, torch.tensor(X_final_val[:1], dtype=torch.float32),
    device="cpu", warmup=100, repeats=2000,
)
oof_total_ms = oof_feature_ms + oof_forward_ms
oof_summary = summary_df[summary_df["student"] == "Distillation OOF"].iloc[0]

oof_day3_benchmark = {
    "Modèle": "Notre approche — codon/phase + MLP KD OOF",
    "Accuracy": oof_summary["accuracy_moyenne"],
    "F1": oof_summary["f1_moyen"],
    "Paramètres": count_trainable_parameters(benchmark_student),
    "Taille state_dict (MB)": model_storage_mb(benchmark_student),
    "Features (ms/séquence)": oof_feature_ms,
    "Forward CPU (ms/séquence)": oof_forward_ms,
    "Latence totale (ms/séquence)": oof_total_ms,
    "Débit estimé (séquences/s)": 1000.0 / oof_total_ms,
}
display(pd.DataFrame([oof_day3_benchmark]).style.format({
    "Accuracy": "{:.2%}", "F1": "{:.2%}",
    "Taille state_dict (MB)": "{:.4f}",
    "Features (ms/séquence)": "{:.4f}",
    "Forward CPU (ms/séquence)": "{:.4f}",
    "Latence totale (ms/séquence)": "{:.4f}",
    "Débit estimé (séquences/s)": "{:.1f}",
}))

,Modèle,Accuracy,F1,Paramètres,Taille state_dict (MB),Features (ms/séquence),Forward CPU (ms/séquence),Latence totale (ms/séquence),Débit estimé (séquences/s)
0,Notre approche — codon/phase + MLP KD OOF,90.13%,90.52%,48961,0.1986,0.3879,0.0190,0.4069,2457.4


## Conclusion

Dans ce protocole strict, la distillation OOF améliore le petit MLP en moyenne et lui permet de franchir légèrement la cible des 90 %. Le gain d'Accuracy reste modeste et varie selon la graine ; le gain de ROC-AUC est plus net et régulier.

La formulation rigoureuse est : **meilleure parmi les variantes mesurées avec les hyperparamètres choisis sans utiliser la validation officielle**. Ce résultat ne prouve pas un optimum global.

In [11]:
hard_mean = summary_df.loc[summary_df.student == "Sans distillation", "accuracy_moyenne"].iloc[0]
kd_mean = summary_df.loc[summary_df.student == "Distillation OOF", "accuracy_moyenne"].iloc[0]
hard_auc = summary_df.loc[summary_df.student == "Sans distillation", "roc_auc_moyenne"].iloc[0]
kd_auc = summary_df.loc[summary_df.student == "Distillation OOF", "roc_auc_moyenne"].iloc[0]
print(f"Accuracy moyenne témoin : {hard_mean:.2%}")
print(f"Accuracy moyenne KD OOF : {kd_mean:.2%}")
print(f"Gain Accuracy : {(kd_mean - hard_mean) * 100:+.2f} point")
print(f"Gain ROC-AUC : {(kd_auc - hard_auc) * 100:+.2f} point")
print("Test consulté : NON")

Accuracy moyenne témoin : 89.80%
Accuracy moyenne KD OOF : 90.13%
Gain Accuracy : +0.33 point
Gain ROC-AUC : +0.99 point
Test consulté : NON
